## Vector Stores and Retrievers

In [65]:
# !pip install langchain_groq langchain_chroma langchain_huggingface langchain_core

In [66]:
from langchain_core.documents import Document

documents = [

    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),

    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),

    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),

    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    ),

    Document(
        page_content="Rabits are special animals that need plenty of space to hop around.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [67]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Goldfish are popular pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Rabits are special animals that need plenty of space to hop around.')]

In [68]:
from google.colab import userdata

In [69]:
import os
groq_api_key = userdata.get('GROQ_API_KEY')

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [70]:
from langchain_groq import ChatGroq

llm = ChatGroq(groq_api_key = groq_api_key, model = "llama-3.1-8b-instant")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7b4bb1a3a120>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7b4bafea08c0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [71]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [72]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(documents, embedding = embeddings)
vector_store

In [73]:
vector_store.similarity_search("cat")

[Document(id='bc30ce38-aa1d-4788-a3b7-af3cbd8009bd', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='1aa57dca-aba4-46b0-9737-f86c29919836', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='3f85ec1a-a98c-40f8-8c84-b3d6a767c217', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='b574d6dd-d2b6-4277-8aa7-4412e4f0a2f7', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')]

In [74]:
await vector_store.asimilarity_search("cat")

[Document(id='bc30ce38-aa1d-4788-a3b7-af3cbd8009bd', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='1aa57dca-aba4-46b0-9737-f86c29919836', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='3f85ec1a-a98c-40f8-8c84-b3d6a767c217', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='b574d6dd-d2b6-4277-8aa7-4412e4f0a2f7', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')]

In [75]:
vector_store.similarity_search_with_score("cat")

[(Document(id='bc30ce38-aa1d-4788-a3b7-af3cbd8009bd', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.935105562210083),
 (Document(id='1aa57dca-aba4-46b0-9737-f86c29919836', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.935105562210083),
 (Document(id='3f85ec1a-a98c-40f8-8c84-b3d6a767c217', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.935105562210083),
 (Document(id='b574d6dd-d2b6-4277-8aa7-4412e4f0a2f7', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.935105562210083)]

## Retrievers

In [76]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vector_store.similarity_search).bind(k=1)
retriever.batch(["cat","dog"])

[[Document(id='bc30ce38-aa1d-4788-a3b7-af3cbd8009bd', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='30bfdc90-138f-494e-8e88-2b16465b64b2', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [78]:
retriever = vector_store.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k": 1}
)
retriever.batch(["cat","dog"])

[[Document(id='bc30ce38-aa1d-4788-a3b7-af3cbd8009bd', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='30bfdc90-138f-494e-8e88-2b16465b64b2', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [79]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [81]:
message = """
Answer this question using the provided context only.
{question}
Context:
{Context}
"""

In [86]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [87]:
prompt = ChatPromptTemplate.from_messages([("human", message)])

rag_chain = {"Context": retriever, "question": RunnablePassthrough()} | prompt | llm | parser

In [89]:
response = rag_chain.invoke("Tell me about dogs")
print(response)

Dogs are great companions, known for their loyalty and friendliness.
